In [4]:
# pip install ucimlrepo

In [85]:
import os
import pandas as pd
import numpy as np 

from sklearn.model_selection import StratifiedKFold, GridSearchCV , KFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler , MinMaxScaler
from sklearn.compose import make_column_selector, make_column_transformer
import warnings
warnings.filterwarnings("ignore")

In [86]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
cirrhosis_patient_survival_prediction = fetch_ucirepo(id=878) 
  
# data (as pandas dataframes) 
X = cirrhosis_patient_survival_prediction.data.features 
y = cirrhosis_patient_survival_prediction.data.targets 

custom_nans = ["mamm", "nann", "NaNN", "nan"]
X =  X.replace(custom_nans, np.nan)

X['Cholesterol'] = X['Cholesterol'].astype(float)
X['Copper'] = X['Copper'].astype(float)
X['Tryglicerides'] = X['Tryglicerides'].astype(float)
X['Platelets'] = X['Platelets'].astype(float)


In [87]:
 X.dtypes

Drug                 str
Age                int64
Sex                  str
Ascites              str
Hepatomegaly         str
Spiders              str
Edema                str
Bilirubin        float64
Cholesterol      float64
Albumin          float64
Copper           float64
Alk_Phos         float64
SGOT             float64
Tryglicerides    float64
Platelets        float64
Prothrombin      float64
Stage            float64
dtype: object

In [88]:
X

,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage
0,D-penicillamine,21464,F,Y,Y,Y,Y,14.5,261.0,2.60,156.0,1718.0,137.95,172.0,190.0,12.2,4.0
1,D-penicillamine,20617,F,N,Y,Y,N,1.1,302.0,4.14,54.0,7394.8,113.52,88.0,221.0,10.6,3.0
2,D-penicillamine,25594,M,N,N,N,S,1.4,176.0,3.48,210.0,516.0,96.10,55.0,151.0,12.0,4.0
3,D-penicillamine,19994,F,N,Y,Y,S,1.8,244.0,2.54,64.0,6121.8,60.63,92.0,183.0,10.3,4.0
4,Placebo,13918,F,N,Y,Y,N,3.4,279.0,3.53,143.0,671.0,113.15,72.0,136.0,10.9,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,NaN,24472,F,NaN,NaN,NaN,N,1.2,NaN,2.96,NaN,NaN,NaN,NaN,174.0,10.9,3.0
414,NaN,14245,F,NaN,NaN,NaN,N,0.9,NaN,3.83,NaN,NaN,NaN,NaN,180.0,11.2,4.0
415,NaN,20819,F,NaN,NaN,NaN,N,1.6,NaN,3.42,NaN,NaN,NaN,NaN,143.0,9.9,3.0
416,NaN,21185,F,NaN,NaN,NaN,N,0.8,NaN,3.75,NaN,NaN,NaN,NaN,269.0,10.4,3.0


In [89]:
 
scl_scaler , mm_scaler = StandardScaler(), MinMaxScaler()

knn = KNeighborsClassifier(n_neighbors=1)
kfold = KFold(random_state=26, n_splits=5, shuffle=True)

imp_cat  = SimpleImputer(strategy='constant' , fill_value='Unknown').set_output(transform='pandas')
imp_num = SimpleImputer(strategy='mean').set_output(transform='pandas')

ct_impute = make_column_transformer(
                            (imp_cat,           make_column_selector(dtype_include = object)),
                            (imp_num , make_column_selector(dtype_include =['int64','float64'])),
                             verbose_feature_names_out = False).set_output(transform='pandas') 

X_ohe = ct_impute.fit_transform(X)
# X_ohe.shape
X_ohe

,Drug,Sex,Ascites,Hepatomegaly,Spiders,Edema,Age,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage
0,D-penicillamine,F,Y,Y,Y,Y,21464.0,14.5,261.000000,2.60,156.000000,1718.000000,137.950000,172.000000,190.0,12.2,4.0
1,D-penicillamine,F,N,Y,Y,N,20617.0,1.1,302.000000,4.14,54.000000,7394.800000,113.520000,88.000000,221.0,10.6,3.0
2,D-penicillamine,M,N,N,N,S,25594.0,1.4,176.000000,3.48,210.000000,516.000000,96.100000,55.000000,151.0,12.0,4.0
3,D-penicillamine,F,N,Y,Y,S,19994.0,1.8,244.000000,2.54,64.000000,6121.800000,60.630000,92.000000,183.0,10.3,4.0
4,Placebo,F,N,Y,Y,N,13918.0,3.4,279.000000,3.53,143.000000,671.000000,113.150000,72.000000,136.0,10.9,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,Unknown,F,Unknown,Unknown,Unknown,N,24472.0,1.2,369.510563,2.96,97.648387,1982.655769,122.556346,124.702128,174.0,10.9,3.0
414,Unknown,F,Unknown,Unknown,Unknown,N,14245.0,0.9,369.510563,3.83,97.648387,1982.655769,122.556346,124.702128,180.0,11.2,4.0
415,Unknown,F,Unknown,Unknown,Unknown,N,20819.0,1.6,369.510563,3.42,97.648387,1982.655769,122.556346,124.702128,143.0,9.9,3.0
416,Unknown,F,Unknown,Unknown,Unknown,N,21185.0,0.8,369.510563,3.75,97.648387,1982.655769,122.556346,124.702128,269.0,10.4,3.0


In [90]:
ohe = OneHotEncoder(sparse_output = False, drop = 'first').set_output(transform = 'pandas')

ct_ohe = make_column_transformer(
                            (ohe,           make_column_selector(dtype_include = object)),
                            ("passthrough" , make_column_selector(dtype_include =['int64','float64'])),
                             verbose_feature_names_out = False).set_output(transform='pandas') 

In [91]:
pipe = Pipeline([ ("IMP" ,ct_impute ),('OHE',ct_ohe) , ("SCL", scl_scaler), ('KNN', knn)])
pipe.get_params()

{'memory': None,
 'steps': [('IMP',
   ColumnTransformer(transformers=[('simpleimputer-1',
                                    SimpleImputer(fill_value='Unknown',
                                                  strategy='constant'),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000025708F04AD0>),
                                   ('simpleimputer-2', SimpleImputer(),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x000002570925E1B0>)],
                     verbose_feature_names_out=False)),
  ('OHE',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x000002570925ECF0>),
                                   ('passthrough', 'passthr

In [92]:
scl_scaler, mm_scaler = StandardScaler(), MinMaxScaler()
imputer = SimpleImputer()
knn = KNeighborsClassifier()
kfold = KFold(n_splits=5, shuffle=True, random_state=26)
params = {'IMP__simpleimputer-1__strategy': ['constant', 'most-frequent'],
    "IMP__simpleimputer-2__strategy":['mean','median'],
    "SCL":[scl_scaler, mm_scaler, None],
    "KNN__n_neighbors": np.arange(1,11)
}

gcv = GridSearchCV(pipe, param_grid=params, cv=kfold, scoring='neg_log_loss', n_jobs=-1)
gcv.fit(X,y)
print('Best params: ', gcv.best_params_)
print('Best score: ', gcv.best_score_)

Best params:  {'IMP__simpleimputer-1__strategy': 'constant', 'IMP__simpleimputer-2__strategy': 'mean', 'KNN__n_neighbors': np.int64(10), 'SCL': None}
Best score:  -1.7238268147136417


In [83]:
from sklearn.ensemble import RandomForestClassifier 

In [98]:
rfc = RandomForestClassifier(random_state=26)

pipe = Pipeline([ ("IMP" ,ct_impute ),('OHE',ct_ohe) , ('RFC', rfc)])
pipe.get_params()

{'memory': None,
 'steps': [('IMP',
   ColumnTransformer(transformers=[('simpleimputer-1',
                                    SimpleImputer(fill_value='Unknown',
                                                  strategy='constant'),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000025708F04AD0>),
                                   ('simpleimputer-2', SimpleImputer(),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x000002570925E1B0>)],
                     verbose_feature_names_out=False)),
  ('OHE',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x000002570925ECF0>),
                                   ('passthrough', 'passthr

In [101]:
scl_scaler, mm_scaler = StandardScaler(), MinMaxScaler()
imputer = SimpleImputer()
rfc = RandomForestClassifier(random_state=26)

kfold = KFold(n_splits=5, shuffle=True, random_state=26)
params = {'IMP__simpleimputer-1__strategy': ['constant', 'most-frequent'],
    "IMP__simpleimputer-2__strategy":['mean','median'],
    "RFC__max_features": [2,3,4,5,6],
    "RFC__n_estimators": [50,100],
         "RFC__max_depth":[None, 5 , 10] 
}

gcv = GridSearchCV(pipe, param_grid=params, cv=kfold, scoring='neg_log_loss', n_jobs=-1)
gcv.fit(X,y)
print('Best params: ', gcv.best_params_)
print('Best score: ', gcv.best_score_)

Best params:  {'IMP__simpleimputer-1__strategy': 'constant', 'IMP__simpleimputer-2__strategy': 'mean', 'RFC__max_depth': 5, 'RFC__max_features': 5, 'RFC__n_estimators': 100}
Best score:  -0.6496290755083154
